In [1]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.special import logsumexp
import graphical_sampling as gs
import matplotlib.pyplot as plt

In [ ]:
class SinkhornBalancedKMeans:

    def __init__(
        self,
        n_clusters,
        epsilon=0.05,
        max_iter=50,
        sinkhorn_iter=50,
        tol=1e-4,
        n_init=10,
        random_state=None,
    ):
        self.K = n_clusters
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.sinkhorn_iter = sinkhorn_iter
        self.tol = tol
        self.n_init = n_init
        self.random_state = random_state


    def _compute_distances(self, X, centers):
        return cdist(X, centers, metric="sqeuclidean")


    def _weighted_kmeanspp(self, X, pi, rng):

        N, D = X.shape
        K = self.K

        centers = np.empty((K, D))

        idx = rng.integers(N)
        centers[0] = X[idx]

        closest_dist = np.sum((X - centers[0]) ** 2, axis=1)

        for k in range(1, K):

            probs = pi * closest_dist
            probs /= probs.sum()

            idx = rng.choice(N, p=probs)
            centers[k] = X[idx]

            new_dist = np.sum((X - centers[k]) ** 2, axis=1)
            closest_dist = np.minimum(closest_dist, new_dist)

        return centers


    def _sinkhorn_log(self, C, pi, b, f, g):

        eps = self.epsilon

        for _ in range(self.sinkhorn_iter):

            tmp = (g[None, :] - C) / eps
            f = eps * (np.log(pi) - logsumexp(tmp, axis=1))

            tmp = (f[:, None] - C) / eps
            g = eps * (np.log(b) - logsumexp(tmp, axis=0))

        return f, g


    def _hard_objective(self, X, centers, labels, pi):

        d = np.sum((X - centers[labels]) ** 2, axis=1)
        return np.sum(pi * d)


    def _fit_single(self, X, pi, rng):

        N, D = X.shape
        K = self.K

        b = np.ones(K)

        centers = self._weighted_kmeanspp(X, pi, rng)

        f = np.zeros(N)
        g = np.zeros(K)

        for _ in range(self.max_iter):

            C = self._compute_distances(X, centers)

            C = C / (np.median(C) + 1e-12)

            f, g = self._sinkhorn_log(C, pi, b, f, g)

            P = np.exp((f[:, None] + g[None, :] - C) / self.epsilon)

            new_centers = P.T @ X

            shift = np.linalg.norm(new_centers - centers)

            centers = new_centers

            if shift < self.tol:
                break

        R = P / pi[:, None]
        labels = np.argmax(R, axis=1)

        obj = self._hard_objective(X, centers, labels, pi)

        return centers, P, R, labels, obj


    def fit(self, X, pi=None):

        rng = np.random.default_rng(self.random_state)

        N, D = X.shape
        K = self.K

        if pi is None:
            pi = np.ones(N) * (K / N)

        assert np.isclose(pi.sum(), K)

        best_obj = np.inf
        best_state = None

        for _ in range(self.n_init):

            centers, P, R, labels, obj = self._fit_single(X, pi, rng)

            if obj < best_obj:
                best_obj = obj
                best_state = (centers, P, R, labels)

        self.centroids, self.transport_plan, self.assignments, self.labels = best_state
        self.objective_ = best_obj

        return self
